In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]

target_col = None

for col in df.columns:
    if col.lower().replace("_", "").replace(" ", "") in [
        "medhouseval", "medianhousevalue", "target"
    ]:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found. Check dataset columns.")

X = df.drop(columns=[target_col]).select_dtypes(include="number")
y = pd.to_numeric(df[target_col], errors="coerce")

valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("Correlation Matrix:")
print(X.corr().round(2))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=10.0))
])

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)

pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)

train_pred1 = model1.predict(X_train)
train_pred2 = model2.predict(X_train)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "Training MSE": [
        mean_squared_error(y_train, train_pred1),
        mean_squared_error(y_train, train_pred2)
    ],
    "Testing MSE": [
        mean_squared_error(y_test, pred1),
        mean_squared_error(y_test, pred2)
    ],
    "Training R2": [
        r2_score(y_train, train_pred1),
        r2_score(y_train, train_pred2)
    ],
    "Testing R2": [
        r2_score(y_test, pred1),
        r2_score(y_test, pred2)
    ]
})

print("\nModel Performance Comparison:")
print(results.to_string(index=False))

linear_coef = model1.named_steps["regressor"].coef_
ridge_coef = model2.named_steps["regressor"].coef_

coef_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Linear Coefficient": linear_coef,
    "Ridge Coefficient": ridge_coef
})

print("\nCoefficient Comparison:")
print(coef_comparison.to_string(index=False))

print("\nCoefficient Magnitude:")
print("Linear Regression:", np.linalg.norm(linear_coef))
print("Ridge Regression:", np.linalg.norm(ridge_coef))


Correlation Matrix:
            MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  \
MedInc        1.00     -0.12      0.33      -0.06        0.00      0.02   
HouseAge     -0.12      1.00     -0.15      -0.08       -0.30      0.01   
AveRooms      0.33     -0.15      1.00       0.85       -0.07     -0.00   
AveBedrms    -0.06     -0.08      0.85       1.00       -0.07     -0.01   
Population    0.00     -0.30     -0.07      -0.07        1.00      0.07   
AveOccup      0.02      0.01     -0.00      -0.01        0.07      1.00   
Latitude     -0.08      0.01      0.11       0.07       -0.11      0.00   
Longitude    -0.02     -0.11     -0.03       0.01        0.10      0.00   

            Latitude  Longitude  
MedInc         -0.08      -0.02  
HouseAge        0.01      -0.11  
AveRooms        0.11      -0.03  
AveBedrms       0.07       0.01  
Population     -0.11       0.10  
AveOccup        0.00       0.00  
Latitude        1.00      -0.92  
Longitude      -0.92       1.00 

In [ ]:
%pip install pandas numpy scikit-learn
# 1. AIM

To implement Ridge Regression on a suitable multivariate open-access dataset and observe its influence on model overfitting and multicollinearity.

# 2. DATASET USED

**Dataset Name:** California Housing Dataset

**Dataset Path:** `C:\Users\Tanuja\Downloads\california_housing.csv`

The California Housing dataset is a real-world dataset containing housing-related and demographic information about different districts in California.

It is used to predict the median house value based on multiple independent variables.

The standard dataset contains 20,640 records, 8 numerical input features and one continuous target variable.

### Features of the Dataset

| Feature | Description |
|---|---|
| MedInc | Median income in the block group |
| HouseAge | Median house age |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Population of the block group |
| AveOccup | Average number of household members |
| Latitude | Geographical latitude |
| Longitude | Geographical longitude |
| MedHouseVal | Median house value (target) |

**Why is this dataset used?**

1. It contains multiple numerical independent variables.
2. It represents a real-world housing prediction problem.
3. It is suitable for applying Linear Regression and Ridge Regression.
4. It helps observe the effect of regularization on regression coefficients.
5. It allows comparison of training and testing performance.

# 3. THEORY

## A. Regularization

Regularization is a technique used in machine learning to reduce overfitting by adding a penalty term to the model's loss function.

It controls model complexity by restricting the magnitude of model coefficients.

Regularization helps a model generalize better to unseen data.

**Need for Regularization:**

1. Reduces the possibility of overfitting.
2. Controls large regression coefficients.
3. Helps handle multicollinearity.
4. Improves model stability.
5. Can improve prediction performance on unseen data.

## B. Ridge Regression

Ridge Regression is a regularized version of Linear Regression that uses L2 regularization.

It adds a penalty proportional to the sum of the squares of regression coefficients to the ordinary least squares loss function.

Ridge Regression reduces the magnitude of coefficients but generally does not make them exactly zero.

It is particularly useful when independent variables are highly correlated.

### Mathematical Formula

Ridge Regression minimizes:

L = Σ(Yᵢ - Ŷᵢ)² + αΣβⱼ²

Where:

- Yᵢ = Actual value.
- Ŷᵢ = Predicted value.
- βⱼ = Regression coefficients.
- α = Regularization parameter.
- Σ(Yᵢ - Ŷᵢ)² = Sum of Squared Errors.
- αΣβⱼ² = L2 regularization penalty.

The first term minimizes prediction errors, while the second term controls the size of regression coefficients.

## C. L2 Regularization

L2 Regularization adds the square of the coefficients to the loss function.

It penalizes large coefficient values and gradually shrinks them towards zero.

Unlike L1 regularization, L2 regularization generally does not eliminate features by making their coefficients exactly zero.

Ridge Regression uses L2 regularization.

## D. Regularization Parameter (Alpha)

Alpha (α) controls the strength of regularization in Ridge Regression.

- **Alpha = 0:** Equivalent to ordinary Linear Regression.
- **Small Alpha:** Applies a weak penalty.
- **Large Alpha:** Applies a stronger penalty and shrinks coefficients more.
- **Very Large Alpha:** May cause underfitting because the coefficients become excessively small.

In this practical, alpha = 10.0 is used.

The purpose is to observe how regularization changes the coefficients and model performance compared with ordinary Linear Regression.

## E. Difference Between Linear Regression and Ridge Regression

**Linear Regression:**

Linear Regression uses the Ordinary Least Squares method to minimize the sum of squared prediction errors without adding a regularization penalty.

**Ridge Regression:**

Ridge Regression adds an L2 penalty to the loss function to control coefficient magnitude and reduce model variance.

| Linear Regression | Ridge Regression |
|---|---|
| No regularization penalty | Uses L2 regularization |
| Minimizes squared errors | Minimizes squared errors plus penalty |
| Coefficients may become unstable under multicollinearity | Coefficients are generally more stable |
| More susceptible to coefficient variance | Helps reduce coefficient variance |
| May overfit in some situations | Helps control overfitting |

## F. Overfitting

Overfitting occurs when a machine learning model learns the training data too closely, including noise and unnecessary patterns.

An overfitted model performs well on training data but poorly on unseen testing data.

**Characteristics of Overfitting:**

1. Very low training error.
2. Higher testing error.
3. Poor generalization.
4. High model variance.

Ridge Regression helps reduce overfitting by penalizing large coefficient values.

However, it does not guarantee that overfitting will always be eliminated.

## G. Multicollinearity

Multicollinearity occurs when two or more independent variables are highly correlated with each other.

It makes it difficult to identify the individual effect of each independent variable.

It can cause regression coefficients to become unstable and sensitive to small changes in training data.

**Example:**

In the California Housing dataset, AveRooms and AveBedrms may be correlated because the average number of bedrooms is related to the average number of rooms.

**Effects of Multicollinearity:**

1. Makes coefficient estimates unstable.
2. Increases the variance of coefficient estimates.
3. Makes individual feature effects difficult to interpret.
4. Can reduce the reliability of coefficient interpretation.

**How Ridge Regression helps:**

Ridge Regression adds an L2 penalty that shrinks coefficients and makes the estimated coefficients more stable when features are highly correlated.

It does not remove the correlation between the original independent variables.

## H. Feature Scaling

Feature Scaling is the process of transforming numerical attributes into a comparable scale.

It is important for Ridge Regression because the regularization penalty acts on coefficient magnitudes. Features with different scales may otherwise receive unequal effective regularization.

In this practical, StandardScaler is used.

### StandardScaler Formula

Z = (X - μ) / σ

Where:

- X = Original value.
- μ = Mean of the feature.
- σ = Standard deviation.
- Z = Standardized value.

## I. Role of Alpha in Ridge Regression

The alpha parameter controls the strength of the L2 penalty.

As alpha increases:

- Coefficients are generally reduced in magnitude.
- Model complexity is restricted.
- Variance may decrease.
- Bias may increase.

Therefore, choosing an appropriate alpha is important to balance bias and variance.

## J. Evaluation of Ridge Regression

In this practical, Linear Regression and Ridge Regression are compared using the following metrics:

**1. Mean Squared Error (MSE)**

MSE calculates the average squared difference between actual and predicted values.

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

A lower MSE indicates smaller squared prediction errors.

**2. R² Score**

R² measures the proportion of variation in the dependent variable explained by the regression model.

R² = 1 - (SSres / SStot)

A higher R² generally indicates better prediction performance.

**3. Coefficient Magnitude**

Coefficient magnitude indicates the size of regression coefficients.

In this practical, the coefficient magnitudes of Linear Regression and Ridge Regression are compared to observe the shrinking effect of L2 regularization.

## K. Advantages of Ridge Regression

1. Helps reduce overfitting.
2. Handles multicollinearity more effectively than ordinary least squares in many situations.
3. Reduces the magnitude of regression coefficients.
4. Improves model stability.
5. Works well with datasets containing correlated numerical features.

## L. Limitations of Ridge Regression

1. It does not eliminate irrelevant features by making their coefficients exactly zero.
2. It requires choosing an appropriate alpha value.
3. Very high regularization may cause underfitting.
4. It does not completely remove multicollinearity.
5. Feature scaling is important for effective regularization.

# 4. ALGORITHM

1. Start.
2. Import Pandas, NumPy and required Scikit-learn libraries.
3. Load the California Housing dataset using read_csv().
4. Clean the column names and remove unwanted unnamed columns.
5. Identify the target variable.
6. Separate independent variables X and dependent variable y.
7. Remove rows containing missing input or target values.
8. Divide the dataset into training and testing sets.
9. Create a Linear Regression model using LinearRegression().
10. Standardize the input features using StandardScaler().
11. Train the Linear Regression model.
12. Create a Ridge Regression model with alpha = 10.0.
13. Standardize the input features and train the Ridge Regression model.
14. Predict output values using both models.
15. Calculate training and testing MSE.
16. Calculate training and testing R² scores.
17. Compare the regression coefficients of both models.
18. Calculate coefficient magnitudes using the L2 norm.
19. Display the comparison results.
20. Stop.

# 5. DETAILED EXPLANATION OF EACH LINE OF CODE

## A. Importing Libraries

**1. import pandas as pd**

Imports Pandas for loading and manipulating the dataset.

**2. import numpy as np**

Imports NumPy for numerical calculations and coefficient magnitude calculation.

**3. from sklearn.model_selection import train_test_split**

Imports train_test_split() to divide the dataset into training and testing sets.

**4. from sklearn.preprocessing import StandardScaler**

Imports StandardScaler to standardize numerical features before regression.

**5. from sklearn.pipeline import Pipeline**

Imports Pipeline to combine feature scaling and regression into one workflow.

**6. from sklearn.linear_model import LinearRegression, Ridge**

Imports two regression algorithms:

- LinearRegression implements ordinary least squares regression.
- Ridge implements regression with L2 regularization.

**7. from sklearn.metrics import mean_squared_error, r2_score**

Imports the regression evaluation functions:

- mean_squared_error() calculates MSE.
- r2_score() calculates R².

## B. Loading the Dataset

`df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")`

- read_csv() reads the CSV file.
- df stores the dataset in a DataFrame.
- The specified path identifies the location of the dataset.

## C. Cleaning Column Names

`df.columns = df.columns.astype(str).str.strip()`

Converts column names to strings and removes extra spaces.

`df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]`

Removes unwanted unnamed columns that may be created during CSV export.

## D. Identifying the Target Variable

`target_names = ["medhouseval", "medianhousevalue", "target"]`

Creates a list containing common names of the target variable.

`target_col = next(...)`

Searches the dataset and identifies the target column.

`if target_col is None:`

Checks whether a valid target column was found.

`raise ValueError(...)`

Displays an error if the target column is missing.

## E. Selecting Independent Variables

`X = df.drop(columns=[target_col]).select_dtypes(include="number")`

- drop() removes the target column from the input dataset.
- select_dtypes(include="number") selects numerical features.
- X stores the independent variables.

## F. Selecting Dependent Variable

`y = pd.to_numeric(df[target_col], errors="coerce")`

Converts the target column into numerical form.

`errors="coerce"` converts invalid values into missing values.

The variable y stores the dependent variable.

## G. Removing Missing Values

`valid_rows = X.notna().all(axis=1) & y.notna()`

Checks whether all input features and the target value are available for each observation.

`X = X.loc[valid_rows]`

Keeps valid input records.

`y = y.loc[valid_rows]`

Keeps corresponding target values.

## H. Splitting the Dataset

`train_test_split(X, y, test_size=0.2, random_state=42)`

Divides the dataset into:

- 80% training data.
- 20% testing data.

random_state=42 ensures reproducible splitting.

## I. Creating Linear Regression Model

`model1 = Pipeline([...])`

Creates a pipeline containing:

1. StandardScaler.
2. LinearRegression.

`StandardScaler()`

Standardizes numerical input features.

`LinearRegression()`

Creates a regression model without a regularization penalty.

## J. Creating Ridge Regression Model

`model2 = Pipeline([...])`

Creates a pipeline containing:

1. StandardScaler.
2. Ridge Regression.

`Ridge(alpha=10.0)`

Creates a Ridge Regression model with a regularization strength of 10.

The alpha parameter controls the strength of the L2 penalty.

## K. Training Both Models

`model1.fit(X_train, y_train)`

Trains the Linear Regression model using the training dataset.

`model2.fit(X_train, y_train)`

Trains the Ridge Regression model using the same training dataset.

Both pipelines learn scaling parameters from the training data only.

## L. Making Predictions

`pred1 = model1.predict(X_test)`

Predicts house values using Linear Regression.

`pred2 = model2.predict(X_test)`

Predicts house values using Ridge Regression.

`train_pred1 = model1.predict(X_train)`

Generates training predictions using Linear Regression.

`train_pred2 = model2.predict(X_train)`

Generates training predictions using Ridge Regression.

## M. Creating Performance Comparison

`mean_squared_error(y_train, train_pred1)`

Calculates training MSE for Linear Regression.

`mean_squared_error(y_train, train_pred2)`

Calculates training MSE for Ridge Regression.

`mean_squared_error(y_test, pred1)`

Calculates testing MSE for Linear Regression.

`mean_squared_error(y_test, pred2)`

Calculates testing MSE for Ridge Regression.

Similarly, r2_score() calculates training and testing R² values for both models.

## N. Comparing Regression Coefficients

`linear_coef = model1.named_steps["regressor"].coef_`

Extracts the regression coefficients learned by Linear Regression.

`ridge_coef = model2.named_steps["regressor"].coef_`

Extracts the regression coefficients learned by Ridge Regression.

The coefficient comparison helps observe how regularization shrinks coefficient magnitudes.

## O. Calculating Coefficient Magnitude

`np.linalg.norm(linear_coef)`

Calculates the Euclidean or L2 norm of Linear Regression coefficients.

`np.linalg.norm(ridge_coef)`

Calculates the Euclidean or L2 norm of Ridge Regression coefficients.

A smaller coefficient magnitude generally indicates that Ridge Regression has reduced the size of the coefficients.

# 6. EXPECTED OUTPUT

For the standard California Housing dataset:

- Total records: 20,640.
- Input features: 8.
- Target variable: MedHouseVal.
- Training records: 16,512.
- Testing records: 4,128.
- Ridge regularization parameter: 10.0.

The program displays:

1. Model Performance Comparison.
2. Training MSE of both models.
3. Testing MSE of both models.
4. Training R² of both models.
5. Testing R² of both models.
6. Linear Regression coefficients.
7. Ridge Regression coefficients.
8. Coefficient magnitude of both models.

The actual values are calculated during execution and may vary depending on the contents of your teacher-provided CSV.

# 7. ORAL EXAMINATION – VIVA QUESTIONS AND ANSWERS

## Basic Questions

**Q1. What is Regularization?**

Regularization is a technique used to control model complexity by adding a penalty term to the loss function.

**Q2. What is Ridge Regression?**

Ridge Regression is a regularized version of Linear Regression that uses L2 regularization.

**Q3. What is L2 Regularization?**

L2 Regularization adds the sum of squared regression coefficients as a penalty to the loss function.

**Q4. Why is Ridge Regression used?**

It helps reduce overfitting, controls large coefficients and improves coefficient stability under multicollinearity.

**Q5. What is Alpha in Ridge Regression?**

Alpha is the regularization parameter that controls the strength of the L2 penalty.

**Q6. What happens when alpha increases?**

The regularization penalty becomes stronger, and coefficient magnitudes generally decrease.

**Q7. What happens when alpha is zero?**

Ridge Regression becomes equivalent to ordinary least squares Linear Regression.

**Q8. What is the difference between Linear Regression and Ridge Regression?**

Linear Regression minimizes squared errors, whereas Ridge Regression minimizes squared errors with an additional L2 penalty.

**Q9. What is a regression coefficient?**

A regression coefficient represents the estimated change in predicted output for a one-unit change in a particular input feature, keeping other features constant.

**Q10. What is the target variable in this practical?**

Median House Value.

## Questions on Overfitting

**Q11. What is Overfitting?**

Overfitting occurs when a model learns training data too closely, including noise, and performs poorly on unseen data.

**Q12. How does Ridge Regression reduce overfitting?**

It penalizes large coefficients and reduces model variance, which can improve generalization.

**Q13. What is the difference between overfitting and underfitting?**

Overfitting occurs when a model is too complex, whereas underfitting occurs when a model is too simple to capture important patterns.

**Q14. How do you identify overfitting?**

By observing low training error and significantly higher testing error.

## Questions on Multicollinearity

**Q15. What is Multicollinearity?**

Multicollinearity occurs when two or more independent variables are highly correlated with each other.

**Q16. What are the effects of Multicollinearity?**

It can make regression coefficient estimates unstable and difficult to interpret.

**Q17. How does Ridge Regression handle Multicollinearity?**

It shrinks coefficient magnitudes and stabilizes coefficient estimates when independent variables are highly correlated.

**Q18. Does Ridge Regression remove Multicollinearity completely?**

No. It reduces the adverse effect of multicollinearity on coefficient estimation but does not remove correlation between original features.

# 8. DIFFICULT EXTERNAL EXAMINER QUESTIONS AND ANSWERS

**Q19. What is the mathematical objective of Ridge Regression?**

It minimizes the sum of squared prediction errors plus the L2 penalty on the regression coefficients.

**Q20. Why is Ridge Regression called L2 Regularization?**

Because it uses the sum of the squares of the regression coefficients in its penalty term.

**Q21. Does Ridge Regression make coefficients exactly zero?**

Generally, no. Ridge Regression shrinks coefficients towards zero but usually does not make them exactly zero.

**Q22. What is the difference between Ridge and Lasso Regression?**

Ridge uses L2 regularization and generally shrinks coefficients towards zero. Lasso uses L1 regularization and can make some coefficients exactly zero.

**Q23. What happens if alpha is extremely large?**

The coefficients become very small, and the model may underfit the data.

**Q24. Why is StandardScaler important before Ridge Regression?**

Because Ridge penalizes coefficient magnitudes. Scaling ensures that different feature measurement scales do not create unequal effective regularization.

**Q25. Why is Ridge Regression useful when features are highly correlated?**

Highly correlated features can make ordinary least squares coefficient estimates unstable. Ridge regularization stabilizes the estimates by shrinking coefficients.

**Q26. What is the difference between Bias and Variance?**

Bias is error due to overly simple model assumptions. Variance is the sensitivity of model predictions to changes in training data.

**Q27. Can Ridge Regression increase bias?**

Yes. Regularization may increase bias while reducing variance.

**Q28. Does Ridge Regression always produce better testing accuracy?**

No. Its performance depends on the dataset and selected alpha value. Excessive regularization may cause underfitting.

**Q29. How can we choose the best alpha value?**

We can test different alpha values using cross-validation and select the value that provides the best validation performance.

**Q30. What is the difference between the L1 and L2 penalties?**

L1 uses absolute coefficient values, while L2 uses squared coefficient values.

**Q31. What is the purpose of comparing coefficient magnitudes?**

It helps observe the shrinkage effect of Ridge regularization on the learned regression coefficients.

**Q32. Why do we compare training and testing MSE?**

To understand whether the model is overfitting and how well it generalizes to unseen data.

**Q33. Can Ridge Regression be used when there are more features than observations?**

Yes. Ridge Regression can provide a regularized solution even when the number of features is greater than the number of observations.

**Q34. What is the difference between ordinary least squares and Ridge Regression?**

Ordinary least squares minimizes the residual sum of squares, while Ridge Regression minimizes the residual sum of squares plus an L2 coefficient penalty.

**Q35. Does Ridge Regression perform feature selection?**

No. Ridge generally retains all features but reduces their coefficient magnitudes.

# 9. DATASET-RELATED QUESTIONS

**Q1. What is the California Housing dataset?**

It is a real-world dataset containing housing and demographic information from California districts.

**Q2. How many records are present in the standard dataset?**

20,640 records.

**Q3. How many numerical input features are present?**

Eight numerical input features.

**Q4. What is the target variable?**

MedHouseVal, which represents median house value.

**Q5. What does MedInc represent?**

Median income in the block group.

**Q6. What does HouseAge represent?**

Median house age in the block group.

**Q7. What does AveRooms represent?**

Average number of rooms per household.

**Q8. What does AveBedrms represent?**

Average number of bedrooms per household.

**Q9. Why is California Housing suitable for Ridge Regression?**

It contains multiple numerical input features that can have correlated relationships, making it suitable for demonstrating coefficient regularization.

**Q10. Why is feature scaling used for this dataset?**

Because the input features have different numerical scales, and Ridge Regression penalizes coefficient magnitudes.

# 10. PRACTICAL EXAMINATION – CODING QUESTIONS

**Q1. How do you load a CSV dataset?**

`pd.read_csv("california_housing.csv")`

**Q2. How do you divide the dataset?**

`train_test_split(X, y, test_size=0.2)`

**Q3. How do you create a Linear Regression model?**

`LinearRegression()`

**Q4. How do you create a Ridge Regression model?**

`Ridge(alpha=10.0)`

**Q5. How do you standardize input features?**

`StandardScaler()`

**Q6. How do you train a Ridge model?**

`model.fit(X_train, y_train)`

**Q7. How do you predict output values?**

`model.predict(X_test)`

**Q8. How do you calculate MSE?**

`mean_squared_error(y_test, y_pred)`

**Q9. How do you calculate R²?**

`r2_score(y_test, y_pred)`

**Q10. How do you access regression coefficients?**

`model.named_steps["regressor"].coef_`

**Q11. How do you calculate coefficient magnitude?**

`np.linalg.norm(coef)`

**Q12. How do you access the intercept?**

`model.named_steps["regressor"].intercept_`

# 11. IMPORTANT FORMULAS

**1. Ridge Regression Equation**

L = Σ(Yᵢ - Ŷᵢ)² + αΣβⱼ²

**2. Standardization**

Z = (X - μ) / σ

**3. Mean Squared Error**

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

**4. R² Score**

R² = 1 - (SSres / SStot)

**5. L2 Coefficient Magnitude**

||β||₂ = √(β₁² + β₂² + ... + βₙ²)

# 12. IMPORTANT POINTS FOR VIVA

1. Ridge Regression uses L2 regularization.
2. It adds a penalty to the sum of squared coefficients.
3. Alpha controls the regularization strength.
4. Higher alpha generally produces stronger coefficient shrinkage.
5. Ridge Regression helps reduce overfitting.
6. It stabilizes regression coefficients under multicollinearity.
7. It does not generally make coefficients exactly zero.
8. StandardScaler is important before applying Ridge Regression.
9. Linear Regression is used as the baseline model in this practical.
10. Training and testing MSE and R² are compared to observe model performance.
11. Coefficient magnitudes are compared to observe the regularization effect.